# 01 - Analisis Exploratorio de Datos (EDA)

Este notebook tiene como objetivo realizar una primera exploracion del dataset de siniestros viales. En esta etapa no se aplican procesos de limpieza, transformacion ni feature engineering; solo se inspecciona la estructura general de los datos.


## 1. Importacion de librerias

Se importan las librerias principales para manipulacion de datos y visualizacion, junto con la funcion de carga definida en `src/data_loader.py`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import (
    GRAVEDAD_VICTIMA_ORDER,
    OFFICIAL_CATEGORY_DEFINITIONS,
    OFFICIAL_DATA_DICTIONARY,
    SD_MEANING,
    SD_VALUE,
    cargar_dataset,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")


## 2. Carga del dataset

El archivo original se carga desde la carpeta `data/raw/` usando `cargar_dataset()`. No se modifica el dataset original.


In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_FILE = RAW_DATA_DIR / "siniestros.xlsx"

df = cargar_dataset(DATA_FILE)

if "df" not in globals():
    raise NameError("No se encontro el DataFrame df. Ejecute primero la celda de carga del dataset.")

TARGET_COL = "GRAVEdad_victima"
if TARGET_COL not in df.columns:
    raise KeyError(f"No se encontro la columna target requerida: {TARGET_COL}")

target = df[TARGET_COL].copy()

print(f"Dataset cargado desde: {DATA_FILE}")
print(f"Variable target definida: {TARGET_COL}")


## 3. Data Understanding / Diccionario de Datos

El analisis exploratorio se apoya en el diccionario oficial del dataset de siniestros viales. Esta metadata permite interpretar cada columna segun su definicion institucional y no solo segun su nombre tecnico. En datos administrativos, esta distincion es clave: una etiqueta puede representar un concepto de dominio, una convencion de registro o una ausencia de informacion.

En este dataset, `SD` significa **Sin Datos**. Por lo tanto, no debe interpretarse como una categoria sustantiva comparable con `AUTO`, `PEATON`, `CONDUCTOR` o cualquier otro valor del dominio. Mantener `SD` visible durante el EDA ayuda a evaluar calidad de datos; convertirlo, excluirlo o imputarlo requiere una decision documentada.

La variable `gravedad_victima` debe tratarse como ordinal, con el orden de severidad `LEVE < GRAVE < MORTAL`. Ese orden expresa jerarquia de gravedad, pero no supone que la distancia entre niveles sea numericamente equivalente.

In [ ]:
diccionario_oficial = pd.DataFrame(
    OFFICIAL_DATA_DICTIONARY.items(),
    columns=["variable", "significado_oficial"],
)

# El archivo fuente puede traer la columna como GRAVEdad_victima; se documenta
# como gravedad_victima para mantener consistencia con el diccionario oficial.
display(diccionario_oficial)

print(f"Valor {SD_VALUE!r}: {SD_MEANING}")

### Definiciones institucionales de `gravedad_victima`

- `LEVE`: personas lesionadas que reciben el alta medica dentro de las 24hs siguientes al siniestro o hechos sin datos sobre la gravedad de las lesiones provocadas.
- `GRAVE`: lesiones que exigen hospitalizacion de al menos 24 hs o atencion especializada, como fracturas, conmocion, shock grave y laceraciones importantes.
- `MORTAL`: victima que fallece dentro de los 30 dias de producido el siniestro vial por causas directa o indirectamente atribuibles al hecho.

Estas categorias tienen semantica real de dominio. Analizarlas como simples textos puede inducir errores, especialmente si se ordenan alfabeticamente, se mezclan con faltantes o se codifican sin respetar su ordinalidad.

## 3. Vista general del dataset

Se revisa la estructura inicial del dataset: dimensiones, primeras filas, nombres de columnas y tipos de datos inferidos.


### Shape


In [ ]:
df.shape


### Primeras 10 filas

Permite observar el formato general de los registros y detectar rapidamente posibles problemas de lectura.


In [ ]:
df.head(10)


### Nombres de columnas


In [ ]:
df.columns.tolist()


### Tipos de datos

Se revisan los tipos inferidos por pandas para identificar variables numericas, categoricas, fechas u otros formatos que requieran revision posterior.


In [ ]:
df.dtypes


## 4. Tabla resumen de columnas

Se resume el tipo de dato, cantidad y porcentaje de nulos, y cantidad de valores unicos por columna.


In [ ]:
resumen_columnas = pd.DataFrame({
    "columna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "cantidad_nulos": df.isna().sum().values,
    "porcentaje_nulos": (df.isna().mean() * 100).round(2).values,
    "cantidad_valores_unicos": df.nunique(dropna=True).values,
})

resumen_columnas


### Analisis ordinal de `gravedad_victima`

Para el analisis de severidad se usa el orden institucional `LEVE < GRAVE < MORTAL`. Esta codificacion se crea como variable derivada dentro del notebook y no modifica el dataset original. Su objetivo es facilitar resumenes, visualizaciones ordenadas y controles de consistencia conceptual.

In [ ]:
orden_gravedad = list(GRAVEDAD_VICTIMA_ORDER.keys())

target_ordinal = (
    target.astype("string")
    .str.strip()
    .map(GRAVEDAD_VICTIMA_ORDER)
)

resumen_ordinal_gravedad = pd.DataFrame({
    "categoria": orden_gravedad,
    "orden_ordinal": [GRAVEDAD_VICTIMA_ORDER[c] for c in orden_gravedad],
    "definicion_institucional": [
        OFFICIAL_CATEGORY_DEFINITIONS["gravedad_victima"][c]
        for c in orden_gravedad
    ],
})

display(resumen_ordinal_gravedad)
print("Registros sin codificacion ordinal:", int(target_ordinal.isna().sum()))

### Riesgos de interpretacion

- Ordenar `gravedad_victima` alfabeticamente altera la jerarquia real de severidad.
- Tratar `SD` como categoria sustantiva puede confundir falta de informacion con un atributo del siniestro.
- Interpretar `LEVE` sin revisar la definicion oficial puede subestimar que la categoria tambien contempla hechos sin datos sobre gravedad de lesiones.
- Una codificacion ordinal habilita analisis de severidad, pero no convierte la escala en intervalar.

## 5. Deteccion inicial de columnas

Se identifican columnas completamente vacias, constantes, posibles identificadores y posibles columnas temporales. Esta deteccion es exploratoria y debe validarse manualmente antes de tomar decisiones.


### Columnas completamente vacias


In [ ]:
columnas_vacias = resumen_columnas.loc[
    resumen_columnas["cantidad_nulos"] == len(df),
    "columna"
].tolist()

columnas_vacias


### Columnas constantes


In [ ]:
columnas_constantes = resumen_columnas.loc[
    resumen_columnas["cantidad_valores_unicos"] <= 1,
    "columna"
].tolist()

columnas_constantes


### Posibles IDs

Se consideran posibles IDs las columnas con nombre asociado a identificadores o con valores unicos para casi todos los registros.


In [ ]:
palabras_id = ("id", "codigo", "cod", "nro", "numero")
proporcion_unicos = df.nunique(dropna=True) / len(df)

posibles_ids = [
    columna
    for columna in df.columns
    if any(palabra in columna.lower() for palabra in palabras_id)
    or proporcion_unicos[columna] >= 0.95
]

posibles_ids


### Columnas datetime

Se detectan columnas temporales por tipo de dato o por nombres que suelen indicar fechas/horas. No se convierten tipos en esta etapa.


In [ ]:
palabras_temporales = ("fecha", "hora", "anio", "ano", "mes", "dia", "periodo")

columnas_datetime = [
    columna
    for columna in df.columns
    if pd.api.types.is_datetime64_any_dtype(df[columna])
    or any(palabra in columna.lower() for palabra in palabras_temporales)
]

columnas_datetime


## 6. Separacion automatica de variables

Se agrupan variables numericas, categoricas y temporales para orientar el analisis posterior. La clasificacion es preliminar y no implica limpieza ni transformacion.


In [ ]:
variables_temporales = columnas_datetime
variables_numericas = [
    columna
    for columna in df.select_dtypes(include=[np.number]).columns.tolist()
    if columna not in variables_temporales
]
variables_categoricas = [
    columna
    for columna in df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    if columna not in variables_temporales
]

print(f"Variables numericas: {len(variables_numericas)}")
print(variables_numericas)

print(f"\nVariables categoricas: {len(variables_categoricas)}")
print(variables_categoricas)

print(f"\nVariables temporales: {len(variables_temporales)}")
print(variables_temporales)


## 7. Registros duplicados

Se calcula la cantidad y el porcentaje de filas duplicadas exactas. En esta etapa solo se informa el valor; no se eliminan registros.


In [ ]:
duplicados = df.duplicated().sum()
porcentaje_duplicados = duplicados / len(df) * 100

print(f"Cantidad de registros duplicados: {duplicados}")
print(f"Porcentaje de registros duplicados: {porcentaje_duplicados:.2f}%")


## 8. Estadisticas descriptivas


### Variables numericas


In [ ]:
if variables_numericas:
    df[variables_numericas].describe().T
else:
    pd.DataFrame(columns=["mensaje"], data=[["No se detectaron variables numericas."]])


### Variables categoricas


In [ ]:
if variables_categoricas:
    df[variables_categoricas].describe().T
else:
    pd.DataFrame(columns=["mensaje"], data=[["No se detectaron variables categoricas."]])


## 9. Primeras observaciones

Completar manualmente luego de ejecutar las celdas anteriores.

- TODO: Registrar dimensiones generales del dataset.
- TODO: Identificar columnas relevantes para el analisis.
- TODO: Se?alar columnas con mayor proporcion de valores nulos.
- TODO: Revisar si existen variables con tipos de datos incorrectos.
- TODO: Indicar si hay duplicados y evaluar su posible impacto.
- TODO: Anotar hipotesis iniciales para explorar en proximos notebooks.


## Posibles variables target

- TODO: Revisar que variable representa mejor el objetivo de prediccion.
- TODO: Evaluar si el target sera una variable de clasificacion o regresion.
- TODO: Confirmar disponibilidad del target al momento de prediccion para evitar fuga de informacion.
- TODO: Documentar criterios de seleccion y descarte de posibles targets.


## 10. Analisis de la variable target

En esta seccion se analiza `gravedad_victima` como posible variable objetivo para un problema de clasificacion supervisada. En el archivo cargado la columna aparece como `GRAVEdad_victima`, por lo que el analisis conserva el nombre original de la fuente, pero la interpretacion sigue el diccionario oficial.


### Conceptos clave

La clasificacion supervisada es una tarea de aprendizaje automatico en la que un modelo aprende a asignar una clase o categoria a cada observacion usando ejemplos historicos donde la respuesta correcta ya es conocida. En este caso, la variable target representa la severidad institucional de la victima.

`gravedad_victima` no es una variable nominal pura: sus categorias tienen orden de dominio (`LEVE < GRAVE < MORTAL`). Esta ordinalidad debe respetarse en tablas, graficos y codificaciones analiticas. Sin embargo, codificarla como 1, 2 y 3 no significa que la distancia entre `LEVE` y `GRAVE` sea equivalente a la distancia entre `GRAVE` y `MORTAL`.

Un dataset desbalanceado aparece cuando una o varias clases tienen muchas mas observaciones que las demas. Esto puede hacer que el modelo aprenda principalmente la clase mayoritaria y tenga bajo desempeno en las clases menos frecuentes. En problemas de seguridad vial, las clases minoritarias suelen ser precisamente las mas criticas desde el punto de vista publico y sanitario.

La metrica accuracy puede ser enganosa en datasets desbalanceados porque un modelo puede obtener un valor alto simplemente prediciendo siempre la clase mayoritaria, aunque falle en los casos mas importantes o minoritarios. Por eso conviene evaluar tambien precision, recall, F1-score y metricas sensibles al orden cuando el objetivo sea la gravedad.


### Valores Ãšnicos y frecuencias


In [ ]:
for c in df.columns:
    print(repr(c))

In [ ]:
TARGET_COL_CANDIDATES = ["gravedad_victima", "GRAVEdad_victima"]
TARGET_COL = next((col for col in TARGET_COL_CANDIDATES if col in df.columns), None)

if TARGET_COL is None:
    raise KeyError(f"No se encontro ninguna columna target en {TARGET_COL_CANDIDATES}.")

target = df[TARGET_COL]

valores_unicos_target = pd.DataFrame({
    "valor_unico": pd.Series(target.unique())
})

frecuencia_absoluta_target = target.value_counts(dropna=False)
frecuencia_porcentual_target = target.value_counts(dropna=False, normalize=True).mul(100).round(2)

resumen_target = (
    pd.concat(
        [
            frecuencia_absoluta_target.rename("frecuencia_absoluta"),
            frecuencia_porcentual_target.rename("frecuencia_porcentual"),
        ],
        axis=1,
    )
    .reset_index()
    .rename(columns={"index": TARGET_COL})
)

print("Valores unicos de la variable target:")
display(valores_unicos_target)

print("Frecuencia absoluta y porcentual:")
display(resumen_target)


### Detecci?n de desbalance, inconsistencias y valores extra?os


In [ ]:
target_no_nulo = target.dropna()
conteos_no_nulos = target_no_nulo.value_counts()
porcentajes_no_nulos = target_no_nulo.value_counts(normalize=True).mul(100)

cantidad_nulos = int(target.isna().sum())
porcentaje_nulos = round(target.isna().mean() * 100, 2)

if len(conteos_no_nulos) > 0:
    clase_mayoritaria = conteos_no_nulos.idxmax()
    frecuencia_mayoritaria = int(conteos_no_nulos.max())
    porcentaje_mayoritario = round(float(porcentajes_no_nulos.max()), 2)
    frecuencia_minoritaria = int(conteos_no_nulos.min())
    porcentaje_balanceo = round((frecuencia_minoritaria / frecuencia_mayoritaria) * 100, 2)
else:
    clase_mayoritaria = None
    frecuencia_mayoritaria = 0
    porcentaje_mayoritario = 0.0
    frecuencia_minoritaria = 0
    porcentaje_balanceo = 0.0

clases_baja_frecuencia = porcentajes_no_nulos[porcentajes_no_nulos < 1].round(2)
dataset_desbalanceado = porcentaje_balanceo < 50 if len(conteos_no_nulos) > 1 else True

target_texto = target.dropna().astype(str)
valores_con_espacios = sorted(target_texto[target_texto.str.strip().ne(target_texto)].unique().tolist())
cantidad_vacios = int(target_texto.str.strip().eq("").sum())
variantes_por_normalizacion = target_texto.groupby(target_texto.str.strip().str.lower()).nunique()
posibles_variantes = variantes_por_normalizacion[variantes_por_normalizacion > 1].index.tolist()

detectores_target = pd.DataFrame([
    {
        "control": "clases_desbalanceadas",
        "resultado": "si" if dataset_desbalanceado else "no",
        "detalle": f"balance min/max = {porcentaje_balanceo}%",
    },
    {
        "control": "valores_nulos",
        "resultado": "si" if cantidad_nulos > 0 else "no",
        "detalle": f"{cantidad_nulos} registros ({porcentaje_nulos}%)",
    },
    {
        "control": "valores_vacios",
        "resultado": "si" if cantidad_vacios > 0 else "no",
        "detalle": f"{cantidad_vacios} registros no nulos con texto vacio",
    },
    {
        "control": "espacios_extra",
        "resultado": "si" if valores_con_espacios else "no",
        "detalle": valores_con_espacios if valores_con_espacios else "sin casos detectados",
    },
    {
        "control": "variantes_por_mayusculas_o_espacios",
        "resultado": "si" if posibles_variantes else "no",
        "detalle": posibles_variantes if posibles_variantes else "sin casos detectados",
    },
    {
        "control": "clases_menores_al_1_por_ciento",
        "resultado": "si" if not clases_baja_frecuencia.empty else "no",
        "detalle": clases_baja_frecuencia.to_dict() if not clases_baja_frecuencia.empty else "sin casos detectados",
    },
])

display(detectores_target)


### Visualizaci?n de la distribuci?n del target


In [ ]:
target_plot = pd.DataFrame({
    "target": target.astype("object").where(target.notna(), "Missing")
})
orden_target = target_plot["target"].value_counts().index

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=target_plot, x="target", order=orden_target, ax=ax, color="#4C78A8")
ax.set_title("Frecuencia absoluta de la variable target")
ax.set_xlabel(TARGET_COL)
ax.set_ylabel("Frecuencia absoluta")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
target_porcentual_plot = (
    target_plot["target"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("target")
    .reset_index(name="porcentaje")
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=target_porcentual_plot, x="target", y="porcentaje", ax=ax, color="#F58518")
ax.set_title("Frecuencia porcentual de la variable target")
ax.set_xlabel(TARGET_COL)
ax.set_ylabel("Porcentaje")
ax.tick_params(axis="x", rotation=30)
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
plt.tight_layout()
plt.show()


### Conclusiones autom?ticas


In [ ]:
cantidad_clases = int(target.nunique(dropna=True))

conclusiones_target = pd.DataFrame([
    {"metrica": "cantidad_de_clases", "valor": cantidad_clases},
    {"metrica": "clase_mayoritaria", "valor": clase_mayoritaria},
    {"metrica": "porcentaje_clase_mayoritaria", "valor": f"{porcentaje_mayoritario}%"},
    {"metrica": "porcentaje_de_balanceo_min_vs_max", "valor": f"{porcentaje_balanceo}%"},
    {"metrica": "dataset_desbalanceado", "valor": "si" if dataset_desbalanceado else "no"},
])

display(conclusiones_target)

print(f"La variable target tiene {cantidad_clases} clases no nulas.")
print(f"La clase mayoritaria es '{clase_mayoritaria}' con {porcentaje_mayoritario}% de los registros no nulos.")
print(f"El porcentaje de balanceo entre la clase minoritaria y la mayoritaria es {porcentaje_balanceo}%.")

if dataset_desbalanceado:
    print("Conclusi?n: la variable target presenta se?ales de desbalance. Para modelado, conviene priorizar precision, recall y F1-score adem?s de accuracy.")
else:
    print("Conclusi?n: no se detecta un desbalance fuerte con el criterio min/max usado en esta revisi?n inicial.")


### Nota sobre transformaciones exploratorias

Las siguientes celdas revisan `edad_victima` y reemplazan `SD` por `NaN` solo dentro de la sesion del notebook para inspeccion descriptiva. No se modifica el archivo original en `data/raw/` ni se altera ningun pipeline del proyecto.

In [ ]:
df["edad_victima"].value_counts(dropna=False).head(30)

In [ ]:
df["edad_victima"] = (
    df["edad_victima"]
    .replace("SD", np.nan)
)

In [ ]:
df["edad_victima"] = pd.to_numeric(
    df["edad_victima"],
    errors="coerce"
)

In [ ]:
df["edad_victima"].describe()

In [ ]:
df["GRAVEdad_victima"].value_counts(dropna=False)